# Current version : 10.C (2025-07-23)

# Libraries and directory (always run)

In [ ]:
### import necessary libraries
from datetime import datetime
# import geopandas as gpd
# from IPython.display import display
import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import math
import numpy as np
import os
from IPython.display import clear_output
import pandas as pd
import seaborn as sns
import scanpy as sc
import random
from sklearn.cluster import KMeans
# from sklearn.metrics import adjusted_rand_score
# from sklearn.neighbors import NearestNeighbors
# from sklearn.neighbors import KNeighborsClassifier
import warnings


pd.options.display.max_rows = 2000

# warnings.filterwarnings("ignore") 
# sc.logging.print_header()
# sc.set_figure_params(facecolor="white", figsize=(4, 10))
# sc.settings.verbosity = 1 # errors (0), warnings (1), info (2), hints (3)
# plt.rcParams["font.family"] = "Arial"
# sns.set_style("white")

start_time = datetime.now()

mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color= ['#009736','#EE2A35',"#3f488aff",
                                                    "#f79a00ff", "#cf1100ff", "#81a5bfff",
                                                    "#f9bd00ff","#547200ff", "#bfd8cdff"]) 

def print_with_elapsed_time(message):
    elapsed_time = datetime.now() - start_time
    elapsed_seconds = elapsed_time.total_seconds()
    print(f"[{elapsed_seconds:.2f} seconds] {message}")

In [ ]:
# print(f"geopandas version: {gpd.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"scanpy version: {sc.__version__}")
# print(f"plt version: {plt.__version__}")

In [ ]:
### Directory where the data is stored

# dir = "/mnt/d/Xenium" #Ubuntu
# dir = "/media/volume/data/spatial/hugo/data" #Ubuntu
# dir = "/media/volume/data/spatial/hugo/data/k5" #Ubuntu
# dir = "D:\\Xenium"

# dir_notebook = '/mnt/d/Jupyter_notebook/Xenium_jupyter_notebook'
dir_notebook = '/media/volume/volume_spatial/hugo/notebook'
dir_notebook = 'D:\\Jupyter_notebook/Xenium_jupyter_notebook'


In [ ]:
from module.misc import sample_name_import

name_dir = "circa-SD"

samples, samples_ids = sample_name_import(name_dir)

print(len(samples))
print(samples)

# Data import

In [ ]:
# adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_Banksy_clusters_combined.h5ad.gz")
# adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_Banksy_annotated_automap.h5ad.gz")
# adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_Banksy_annotated_combined.h5ad.gz")

adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz")
# adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_Banksy_annotated_combined.h5ad.gz")

In [ ]:
df = pd.read_parquet(f'{dir_notebook}/csv/{name_dir}/{name_dir}_norm_combined.parquet')

In [ ]:
df = pd.DataFrame(data=adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)
df.shape

from module.xenium_preprocessing import add_annotations

df = add_annotations(adata, df)

In [ ]:
df.groupby(['run','ZT'])['cell_type_final'].value_counts(dropna=False)#.sort_index().to_csv('data/celltype_run_ZT.csv')
# df.groupby(['run','ZT'])['region_automap_name'].value_counts().sort_index().to_csv('data/region_run_ZT.csv')

In [ ]:
df.groupby('cell_type_final').count()

In [ ]:
if 'leiden_colors' in adata.obs:
    adata.obs = adata.obs.drop(columns=['leiden_colors'])
adata.write_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz")

# Visualization

## UMAP

In [ ]:
if 'reduced_pc_20_umap' not in adata.obsm.keys():
    adata.obsm['reduced_pc_20_umap'] = adata.obsm['X_umap']

In [ ]:
adata.obs.columns

In [ ]:
from module.dataviz_analysis import umap_plot_indi_multi

umap_plot_indi_multi(adata,
                     dir_notebook=dir_notebook,
                     cluster_to_use = "cell_type_newnum_final", ### Use the column with numbers, not with actual name
                     individual_plot = False,
                     save_plot = True,
                     cmap_ = 'tab20',
                     name_dir = name_dir,
                     )

In [ ]:
adata.obs['umap-1'] = adata.obsm['X_umap'][:, 0]
adata.obs['umap-2'] = adata.obsm['X_umap'][:, 1]

In [ ]:
### Generate a color palette for the clusters - to make color stay consistent across samples
num_clusters = len(adata.obs['cell_type_newnum_final'].astype(int).unique())
palette = sns.color_palette("hls", n_colors=num_clusters)
adata.obs['leiden_colors'] = adata.obs['cell_type_newnum_final'].astype(int).apply(lambda x: palette[x])

In [ ]:
### UMAP plot with gene expression as color scale
 
sc.pl.umap(adata, color=["Per1"], vmin = 0)

#### 3D Umap ###TODO: confirm it works

In [ ]:
start_time = datetime.now()
sc.tl.umap(adata, min_dist= 1, n_components=3)
print_with_elapsed_time(f"umap done")
sc.pl.umap(adata, projection='3d')

In [ ]:
import matplotlib.pyplot as plt

adata.obs['umap-1'] = adata.obsm['X_umap'][:, 0]
adata.obs['umap-2'] = adata.obsm['X_umap'][:, 1]
adata.obs['umap-3'] = adata.obsm['X_umap'][:, 2]

fig = plt.figure(figsize=(20,20))
ax = fig.add_subplot(projection='3d')


ax.scatter(adata.obs['umap-1'], adata.obs['umap-2'], adata.obs['umap-3'], c = adata.obs['leiden_colors'], s=0.5, label=adata.obs['cell_type_final'])

ax.legend()
ax.set_xlabel('X Label')
ax.set_ylabel('Y Label')
ax.set_zlabel('Z Label')


In [ ]:
cluster_to_use = 'cell_type_final'
cluster_centroids = adata.obs.groupby(cluster_to_use)[['umap-1', 'umap-2','umap-3']].median()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

adata.obs['umap-1'] = adata.obsm['X_umap'][:, 0]
adata.obs['umap-2'] = adata.obsm['X_umap'][:, 1]
adata.obs['umap-3'] = adata.obsm['X_umap'][:, 2]

cluster_centroids = adata.obs.groupby('cell_type_newnum_final')[['umap-1', 'umap-2','umap-3']].median()

cell_type_unique = adata.obs['cell_type_final'].unique()

fig, ax = plt.subplots(figsize=(15, 10))
ax = fig.add_subplot(projection='3d')

for idx, celltype in enumerate(cell_type_unique):
    adata_sel = adata[(adata.obs[cluster_to_use] == celltype)]
    ratio_ = int(len(adata_sel.obs['cell_id']) / 100)
    adata_sel = adata_sel[0:ratio_]
    celltype_combine = str(idx) +'_' + str(celltype)
    ax.scatter(adata_sel.obs['umap-1'], adata_sel.obs['umap-2'], adata_sel.obs['umap-3'], s=3, c=adata_sel.obs['leiden_colors'], label = celltype_combine)
for cluster_id, centroid in cluster_centroids.iterrows():
    ax.text(centroid['umap-1'], centroid['umap-2'], centroid['umap-3'], str(cluster_id), color='red', fontsize=20, ha = 'center')

plt.legend(markerscale=5, scatterpoints=10, bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)

# if save_plot == True:
#     plt.savefig(f"plot/{name_dir}/{name_dir}_UMAP_all.png")



# fig = plt.figure(figsize=(20,20))
# ax = fig.add_subplot(projection='3d')

# ax.scatter(adata.obs['umap-1'], adata.obs['umap-2'], adata.obs['umap-3'], c = adata.obs['leiden_colors'], s=0.5, label=adata.obs['cell_type_final'])
# ax.legend()
# ax.set_xlabel('X Label')
# ax.set_ylabel('Y Label')
# ax.set_zlabel('Z Label')

## Cluster plot

In [ ]:
adata.obs.columns

In [ ]:
from module.dataviz_analysis import cluster_plot

cluster_plot(adata,
             name_dir= name_dir,
             dir_notebook = dir_notebook,
             cluster_to_use = 'cell_type_newnum_final',
             cluster_to_map = 'all',
             cmap_ = 'tab20b',
             save_plot = True,
            )

## Polygon plots

### Data prep

In [ ]:
def polygonplot_dataprep(adata_main, sample_to_plot, cluster_to_use = 'cell_type_newnum_final', cmap_ = 'tab20b'):

    ### Generate a color palette for the clusters - to make color stay consistent across samples
    num_clusters = len(adata_main.obs[cluster_to_use].astype(int).unique())
    palette = sns.color_palette(cmap_, n_colors=num_clusters)
    adata_main.obs['leiden_colors'] = adata_main.obs[cluster_to_use].astype(int).apply(lambda x: palette[x])

    all_samples = np.array(adata_main.obs['sample'].unique())
    sample_position = np.where(all_samples == sample_to_plot)
    sample_position = sample_position[0][0]

    adata_plot = adata_main[adata_main.obs['sample']==all_samples[sample_position]]
    
    cells_geo = gpd.read_file(f'{dir_notebook}/coordinates/polygons/{all_samples[sample_position]}_cells.geojson')
    cells_geo['centroid'] = cells_geo['geometry'].centroid
    cells_geo['x_coor'] = cells_geo['centroid'].x
    cells_geo['y_coor'] = cells_geo['centroid'].y

    if 'objectType' in cells_geo.columns:
        cells_geo = cells_geo[cells_geo['objectType']=='cell']


    # cluster_dict_region = dict(zip(adata_main.obs['cell_id'], adata_main.obs['region_manual_name']))
    cluster_dict_region_a = dict(zip(adata_main.obs['cell_id'], adata_main.obs['region_automap_name']))
    cluster_dict_leiden = dict(zip(adata_main.obs['cell_id'], adata_main.obs['leiden_colors']))
    # cluster_dict = dict(zip(adata_main.obs['cell_id'], adata_main.obs['cell_type_newnum_final']))
    cluster_dict_type = dict(zip(adata_main.obs['cell_id'], adata_main.obs['cell_type_final']))

    if 'circascore' in adata_main.obs.columns:
        cluster_dict_circascore = dict(zip(adata_main.obs['cell_id'], adata_main.obs['circascore']))
        cells_geo['circascore'] = cells_geo['cell'].map(cluster_dict_circascore)

    cells_geo['leiden_colors'] = cells_geo['cell'].map(cluster_dict_leiden)
    cells_geo['cell type'] = cells_geo['cell'].map(cluster_dict_type)
    # cells_geo['region_manual_name'] = cells_geo['cell'].map(cluster_dict_region)
    cells_geo['region_automap_name'] = cells_geo['cell'].map(cluster_dict_region_a)
    cells_geo = cells_geo.dropna(subset=['region_automap_name'])

    df = pd.DataFrame(data=adata_plot.X.toarray(), index=adata_plot.obs_names, columns=adata_plot.var_names)
    df['cell_id'] = df.index
    mapping_dict_region = dict(zip(adata_plot.obs['cell_id'], adata_plot.obs['region_automap_name']))
    mapping_dict_celltype = dict(zip(adata_plot.obs['cell_id'], adata_plot.obs['cell_type_final']))
    mapping_dict_manos = dict(zip(adata_plot.obs['cell_id'], adata_plot.obs['sample']))

    # # # Use .map() function to rename cell contents in 'col1' based on mapping dictionary
    df['region_automap'] = df['cell_id'].map(mapping_dict_region)
    df['cell_type_final'] = df['cell_id'].map(mapping_dict_celltype)
    df['sample'] = df['cell_id'].map(mapping_dict_manos)
    df.dropna(subset=['cell_type_final'], inplace=True)

    return df, cells_geo, cluster_to_use

In [ ]:
#####################################################################################################################
### This section requires that the cells polygons were extracted and saved as GEOjson (see Polygon_plot2.ipynb)######
#####################################################################################################################

import geopandas as gpd
# from module.dataviz_analysis import polygonplot_dataprep

df, cells_geo, cluster_to_use = polygonplot_dataprep(adata,
                                    sample_to_plot='circa2-ZT01',
                                    cluster_to_use = 'cell_type_newnum_final',
                                    cmap_ = 'tab20',
                                    )

In [ ]:
df.head(2)

In [ ]:
cells_geo.head(2)

### Celltype + genes as symbols

In [ ]:
from module.dataviz_analysis import polygonplot_plot
import pytz

polygonplot_plot(df, cells_geo,
                cluster_to_use = 'cell type',
                gene_ = 'Gad1', # example = 'Gfap'
                # region_only = "HIPP", # example : 'SCH' ### Priority 1
                # region_ = 'STR', # example : 'SCH' ### Priority 2
                # coord_ = None, # example : [1000,2000,2000,3000] ### Priority 3
                coord_ = [5500,6000,4600,5200],
                save_plot = False)


In [ ]:
cells_geo['region_automap_name'].unique()

In [ ]:
import matplotlib.colors

region_ = 'STR'
xmin = cells_geo[cells_geo['region_automap_name']== region_]['x_coor'].min()
xmax = cells_geo[cells_geo['region_automap_name']== region_]['x_coor'].max()
ymin = cells_geo[cells_geo['region_automap_name']== region_]['y_coor'].min()
ymax = cells_geo[cells_geo['region_automap_name']== region_]['y_coor'].max()

cells_geo_crop = cells_geo[(cells_geo['x_coor'] >= xmin) & (cells_geo['x_coor'] <= xmax)  
                            & (cells_geo['y_coor'] >= ymin) & (cells_geo['y_coor'] <= ymax)]


fig, ax = plt.subplots(
    figsize=(20,20)
)   


cmap = plt.cm.rainbow
norm = matplotlib.colors.Normalize(vmin=0, vmax=11)

cells_geo_crop.plot( ax=ax,
                color = cmap(norm(cells_geo_crop.circascore.values)),
                alpha=1,
                aspect=1,
                zorder=1,
                # edgecolor = cells_geo_crop['circascore'],
                )

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
fig.colorbar(sm)

### Gene expression as color gradient

In [ ]:
from module.dataviz_analysis import polygonplot_plot_gradient

polygonplot_plot_gradient(df, cells_geo,
                          gene_ = 'Per1', ## Required
                          region_ = 'HIPP',
                          region_only = None,
                          coord_ = [2000,4000,2000,4000],
                          cmap_ = 'brg',
                          save_plot = False)


## Other plots

In [ ]:
### Plot gene expression

import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style="white")

gene_ = 'Chat'
adata_temp = adata
df_dict = dict(zip(df.index, df[gene_]))
adata_temp.obs[gene_] = adata_temp.obs['cell_id'].map(df_dict)

### to crop
adata_temp = adata_temp[(adata_temp.obs['x_centroid'] > 4000) & (adata_temp.obs['x_centroid'] < 6000)
                            & (adata_temp.obs['y_centroid'] > 4000) & (adata_temp.obs['y_centroid'] < 5000)]

# f, ax = plt.subplots(figsize=(15, 10))

fig, axs = plt.subplots(3,2#,figsize=(15, 15)
                        )
axs = axs.flatten()# Mapping of clusters

for idx, sample in enumerate(samples_ids):
    adata_graph = adata_temp[adata_temp.obs['sample'] == sample]
    
    sns.scatterplot(x='x_centroid', y='y_centroid',
                s=2, legend= False,
                palette='viridis',
                hue=gene_,
                data=adata_graph.obs, ax=axs[idx]).set(title=f"{sample} - {gene_}", xlabel = None, ylabel = None, xticklabels = [],yticklabels = [])

# Create the colorbar
norm = mpl.colors.Normalize(vmin=adata_temp.obs[gene_].min(),vmax=adata_temp.obs[gene_].max())
sm = mpl.cm.ScalarMappable(cmap='viridis', norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axs[-1], aspect=40, shrink=0.8)  # Adjust aspect/shrink

del adata_temp, adata_graph


### Gene expression accross samples

In [ ]:
### Gene expression accross samples
if 'df' not in globals():
    df = pd.DataFrame(data=adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)
    df['cell_id'] = df.index

dict_ = dict(zip(adata.obs['cell_id'],adata.obs['sample']))
df['sample'] = df['cell_id'].map(dict_)

gene = 'Per2'
x = df.groupby('sample')[gene].mean()

plt.scatter(x = x.index, y=x)
plt.tick_params(rotation=90)
plt.ylim(0,)
plt.title(f'Ndn expression of {gene}')
plt.ylabel('Normalized expression')

In [ ]:
# to_use = 'total_transcript' ### Run1
to_use = 'transcript_counts' ### Run3
samples = adata.obs['sample'].unique()

### Map log transcript counts 
adata_sel = adata[(adata.obs['sample'] == samples[0])]
adata_sel.obs[to_use] = adata_sel.obs[to_use].astype(float)
adata_sel.obs['log_transcript_counts'] = adata_sel.obs[to_use].apply(lambda x: math.log10(x))

fig, ax = plt.subplots(figsize=(10,6))
transcript_counts_unique = adata_sel.obs['log_transcript_counts'].unique()
cmap = plt.cm.jet
for cluster_id in transcript_counts_unique:
    cluster_data = adata_sel.obs[adata_sel.obs['log_transcript_counts'] == cluster_id]
    colors = cmap((cluster_id - transcript_counts_unique.min()) / (transcript_counts_unique.max() - transcript_counts_unique.min()))
    plt.scatter(cluster_data['x_centroid'], cluster_data['y_centroid'], color=colors, s=1, label=cluster_id)
plt.xlabel('x_centroid')
plt.ylabel('y_centroid')
plt.title('Map of transcript counts (log)')

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap)
sm.set_array(transcript_counts_unique)
divider = make_axes_locatable(ax)
cbar_ax = divider.append_axes('right', size='1.5%', pad=0.05)
plt.colorbar(sm, cax=cbar_ax, label='Transcript Counts')
plt.ylabel('Transcript counts (log)')

# plt.savefig(f"/mnt/d/Jupyter_notebook/Xenium_jupyter_notebook/plot/{name_dir}/{name_dir}_logtranscounts.png")

In [ ]:
# to_use = 'n_genes' ### Run1
to_use = 'n_genes_by_counts' ### Run3
samples = adata.obs['sample'].unique()

### Map log n_genes_by_counts 
adata_sel = adata[(adata.obs['sample'] == samples[0])]
adata_sel.obs[to_use] = adata_sel.obs[to_use].astype(float)
adata_sel.obs['log_n_genes_by_counts'] = adata_sel.obs[to_use].apply(lambda x: math.log10(x))

fig, ax = plt.subplots(figsize=(10,6))
transcript_counts_unique = adata_sel.obs['n_genes_by_counts'].unique()
cmap = plt.cm.jet
for cluster_id in transcript_counts_unique:
    cluster_data = adata_sel.obs[adata_sel.obs['n_genes_by_counts'] == cluster_id]
    colors = cmap((cluster_id - transcript_counts_unique.min()) / (transcript_counts_unique.max() - transcript_counts_unique.min()))
    plt.scatter(cluster_data['x_centroid'], cluster_data['y_centroid'], color=colors, s=1, label=cluster_id)
plt.xlabel('x_centroid')
plt.ylabel('y_centroid')
plt.title('Map of Nb of gene per cell')

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap)
sm.set_array(transcript_counts_unique)
divider = make_axes_locatable(ax)
cbar_ax = divider.append_axes('right', size='1.5%', pad=0.05)
plt.colorbar(sm, cax=cbar_ax)
plt.ylabel('Nb of gene per cell')

# plt.savefig(f"/mnt/d/Jupyter_notebook/Xenium_jupyter_notebook/plot/{name_dir}/{name_dir}_nbgenes.png")

In [ ]:
### Map mmc:cluster_correlation_coefficient
adata_sel = adata[(adata.obs['sample'] == samples[0])]

fig, ax = plt.subplots(figsize=(10,6))
transcript_counts_unique = adata_sel.obs['mmc:class_correlation_coefficient'].unique()
cmap = plt.cm.jet
for cluster_id in transcript_counts_unique:
    cluster_data = adata_sel.obs[adata_sel.obs['mmc:class_correlation_coefficient'] == cluster_id]
    colors = cmap((cluster_id - transcript_counts_unique.min()) / (transcript_counts_unique.max() - transcript_counts_unique.min()))
    plt.scatter(cluster_data['x_centroid'], cluster_data['y_centroid'], color=colors, s=0.5, label=cluster_id)
plt.xlabel('x_centroid')
plt.ylabel('y_centroid')
plt.title('Map of mmc:class_correlation_coefficient')

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap)
sm.set_array(transcript_counts_unique)
divider = make_axes_locatable(ax)
cbar_ax = divider.append_axes('right', size='5%', pad=0.05)
plt.colorbar(sm, cax=cbar_ax, label='correlation_coefficient')

# plt.savefig(f"/mnt/d/Jupyter_notebook/Xenium_jupyter_notebook/plot/{name_dir}/{name_dir}_mmccoef.png")

## Gene expression plots

### Plot individual genes

In [ ]:
df = pd.read_parquet(f'{dir_notebook}/csv/{name_dir}/{name_dir}_norm_combined.parquet')

In [ ]:
df_NS = df[(df['run']=='circa4')]
df_SD = df[(df['run']=='SD1')]
ZT_dict = dict(zip(df_SD['ZT'],df_SD['ZT']))
ZT_dict['ZT13'] = "ZT21"
ZT_dict['ZT21'] = "ZT13"
df_SD['ZT2'] = df_SD['ZT'].map(ZT_dict)
df_SD['ZT'] = df_SD['ZT2']

In [ ]:
gene_name = 'Pou1f1'
plt.plot(df_NS.groupby('ZT')[gene_name].mean(), "bo-", label = 'Normal Sleep' )
plt.plot(df_SD.groupby('ZT')[gene_name].mean(), "go--", label = 'Disrupted Sleep')
plt.title(gene_name)
plt.legend()

In [ ]:
plt.figure(figsize=(3,5))
plt.bar(x = 'NS', height = df_NS[gene_name].mean(), color = '#EE2A35')
plt.errorbar(x='NS',  y = df_NS[gene_name].mean(), yerr=df_NS[gene_name].std()/ math.sqrt(df_NS['ZT'].nunique()), color = 'black')
plt.bar(x = 'SD', height = df_SD[gene_name].mean(), color = '#009736')
plt.errorbar(x='SD',  y = df_SD[gene_name].mean(), yerr=df_SD[gene_name].std()/ math.sqrt(df_SD['ZT'].nunique()), color = 'black')
plt.ylim(0)
plt.title(gene_name)


### Define marker genes

In [ ]:
marker_genes = [
# 10X annotations
# "Acsbg1","Aqp4","Cdh20","Clmn","Gfap","Gli3","Id2","Mapk4","Ntsr2","Pde7b","Rfx4","Rorb","Slc39a12", #Astrocytes
# "Arhgap12","Fibcd1","Sipa1l3","Wfs1", #CA1-ProS
# "2010300C02Rik","Arhgef28","Bcl11b","Bhlhe22","Cabp7","Cpne4","Igfbp4","Necab2","Prdm8","Strip2","Syndig1", #CA2
# "Cpne6","Epha4","Hat1","Neurod6","Npy2r","Nrp2","Shisa6", #CA3
# "Cdh9","Orai2","Prox1","Rasl10a","Tanc1", #DG
# "Acvrl1","Adgrl4","Car4","Cd93","Cldn5","Cobll1","Emcn","Fgd5","Fn1","Kdr","Ly6a","Mecom","Nostrin","Paqr5","Pecam1","Pglyrp1","Slfn5","Sox17","Zfp366", #Endothelial
# "Arhgap25","Cd300c2","Cd53","Cd68","Ikzf1","Laptm5","Siglech","Sla","Spi1","Trem2",#Microglia
# 'Gjc3','Gpr17','Opalin','Sema3d','Sema6a','Sox10','Zfp536', #Oligodendrocytes
# 'Acta2','Ano1','Arhgap6','Carmn','Cspg4','Fos','Gucy1a1','Inpp4b','Nr2f2','Pip5k1b','Plekha2','Pln','Sncg','Sntb1', # Pericytes
# 'Aldh1a2','Col1a1','Col6a1','Cyp1b1','Dcn','Fmod','Gjb2','Igf2','Pdgfra','Ror1',"Slc13a4","Spp1", #VLMC
# "Chat","Crh","Igf1",'Penk','Pthlh','Sorcs3','Thsd7a','Vip', #Vip interneurons

# 'Arntl','Clock','Cry1','Cry2','Nr1d1',"Per1",'Per2','Per3','Rora','Rorb', ## Clock genes
# 'Gfap','Trem2','Cd44','Spp1','Cd68','Igf1','Spi1','Cd300c2','Cd53','Laptm5','Ikzf1','Arhgap25','Opalin','Prox1','Cbln1','Sema3a','Paqr5','Spag16',
# "Vip", "Pkib", "Tmem255a", "Arhgap6", "Chodl", #SCN rank genes analysis
'Strip2',"Shisa6","Chodl", 'Fos','Sdk2', 'Cdh6','Cobll1','Tanc1'
# 'Dner','Gad1','Rasgrf2','Vat1l','Pde7b','Igfbp5','Rorb','Rims3','Tmem255a','Cdh13','Gad2','Rab3b','Parm1','Tle4','Fhod3','Rmst','Vip','Nr2f2','Arhgap6',
# 'Laptm5','Kctd12','Siglech','Trem2','Cd53','Cd68','Cd300c2','Ikzf1','Spi1','Acsbg1','Gfap','Dpy19l1','Unc13c','Arhgap25','Meis2','Dner','Arhgap12','Igfbp5','Ntsr2',
# "Gfap","Rbp4","Trem2","Th","Laptm5","Syt17","Opn3","Spp1","Cd44","Cd53","Igf1","Gjb2",
# "Vip","Prss35","Cd68","Cplx3","Siglech","Ikzf1","Cd300c2","Dcn","Spi1","Pkib","Fos","Angpt1",
# "Igfbp5","Chrm2","Rspo2","Arhgap25","Sst",
# 'Ntsr2'    
]


### Stacked violin plots

In [ ]:
ax = sc.pl.stacked_violin(adata, ### Can be more useful on subset of the data, otherwise "zero values" greatly change the graph
                         marker_genes, ### marker_genes or individual genes (ex: "Dner")
                         groupby='Genotype',
                         dendrogram=False,
                         log=False,
                         )

### Violin plot for individual genes with individual data point (1 graph/gene)

In [ ]:
sc.pl.violin(adata, marker_genes, groupby='Genotype', order = ['WT','APP'],
             jitter = 0.45,
             # log = True,
             # stripplot = False,
            )

### Heatmap

In [ ]:
from module.misc import genes_list

clock_genes = genes_list('clock')
clock_genes.extend(['cell_id', 'cell_type_final','run','sample'])

# adata_sub = adata[:, adata.var_names.isin(clock_genes)].copy()

# df = pd.DataFrame(data=adata_sub.X.toarray(), index=adata_sub.obs_names, columns=adata_sub.var_names)


In [ ]:
import pandas as pd
import scanpy as sc

df = pd.read_parquet('D:\\Jupyter_notebook\Xenium_jupyter_notebook\csv/circa-SD/circa-SD_norm_combined.parquet', columns = clock_genes)

In [ ]:
from module.xenium_preprocessing import add_annotations

df = add_annotations(adata_sub, df)

In [ ]:
sc.pl.heatmap(adata_sub, var_names = clock_genes, groupby='cell_type_final',
               standard_scale='obs')

In [ ]:
grouped = df.groupby('cell_type_final')[clock_genes].mean()
grouped.shape


In [ ]:
order_list = [
    "Oligodendrocyte","Microglia","Endothelial","SMC","Pericyte","VLMC","ABC","Choroid","Ependymal","Tanycyte","OB STR CTX IMN","OPC","Astro TE",
    "SCH Gaba","PVH Glut","HY Glut","SPA Glut","AHN Glut","PVT Glut","PAL STR Gaba Chol","MEA Glut","MPO Glut","BST Glut","LHA Glut","LH Glut",
    "BAC Glut","TRS BAC Glut","MH Glut","AV Glut","VM MD Glut","LD Glut","VP Glut","SMT Glut","CM Glut","RE Glut","PT Glut","AD Glut","RT ZI Gaba",
    "Pvalb Gaba","STR Gaba","Sst Gaba","Sncg Gaba","Vip Gaba","Lamp5 Gaba","L6 IT CTX Glut","L4 5 IT CTX Glut","L2 3 IT RSP Glut","L5 ET CTX Glut",
    "L5 NP CTX Glut","L6b CTX Glut","L6 CT CTX Glut","CLA EPd CTX Glut","L2 3 PIR ENTl Glut","NLOT Glut","L2 3 IT PIR ENTl Glut","COAa PAA MEA Glut",
    "CA3 Glut","CA2 FC IG Glut","CA1 ProS Glut","DG Glut","STR PAL Gaba","LSX Gaba","STR D1 Gaba","STR D2 Gaba",
]

grouped_sort = grouped.reindex(order_list)
grouped_sort.dropna(axis=0, inplace=True)
grouped_sort = grouped_sort.reindex(sorted(grouped_sort.columns), axis=1)
grouped_sort

In [ ]:
gene_list = grouped_sort.columns
cell_types = grouped_sort.index

In [ ]:
plt.figure(figsize=(4,10))
plt.imshow(grouped_sort)
# plt.xticks(range(len(gene_list)), labels = gene_list, rotation = 90)

In [ ]:
sns.clustermap(grouped_sort, cmap = 'viridis', z_score=1, vmin=-1,vmax=1, center=0,
                col_cluster=False,row_cluster=False, cbar = False, cbar_pos=None,figsize=(5, 10),
                )
plt.ylabel(ylabel=None)
plt.savefig('Gallery/heatmap_clockgenes.svg',dpi=300)

# Analysis

## Highest expressed genes

In [ ]:
# adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_Banksy_annotated_automap.h5ad.gz")
adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz")

# if 'leiden_colors' in adata.obs:
#     adata.obs = adata.obs.drop(columns=['leiden_colors'])
# adata.write_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_Banksy_annotated_automap.h5ad.gz")

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=30, show=None, save=None, ax=None, gene_symbols=None, log=False)

In [ ]:
# adata_WT=adata[adata.obs['Genotype']== 'WT']
# adata_APP=adata[adata.obs['Genotype']== 'APP']
grp = adata.obs['run'].unique()

sc.pl.highest_expr_genes(adata[adata.obs['run'] == grp[0]], n_top=30, show=None, save=None, ax=None, gene_symbols=None, log=False)
sc.pl.highest_expr_genes(adata[adata.obs['run'] == grp[1]], n_top=30, show=None, save=None, ax=None, gene_symbols=None, log=False)

## Find marker genes for each cluster

In [ ]:
adata.obs['cell_type_final'] = pd.Categorical(adata.obs['cell_type_final'])
adata.obs['cell_type_final'].cat.categories


In [ ]:
# Obtain cluster-specific differentially expressed genes

# cluster_to_use = 'cell_type_newnum'
# cluster_to_use = 'cell type'
# cluster_to_use = 'cell_type_auto'
# cluster_to_use = 'cell_type_auto_sub'
cluster_to_use = 'cell_type_final'
# cluster_to_use = 'Genotype'

adata.obs[cluster_to_use] = adata.obs[cluster_to_use].astype(str)
sc.tl.rank_genes_groups(adata, groupby=cluster_to_use, method="wilcoxon", tie_correct = True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata,
                                n_genes=1,
                                values_to_plot="logfoldchanges", cmap='bwr',
                                # vmin=-4,
                                # vmax=4,
                                )

### Hierarchical clustering

In [ ]:
cluster_to_use = 'cell_type_final'
sc.tl.dendrogram(adata, groupby = cluster_to_use, n_pcs=None,
                 use_rep=None, var_names=None, use_raw=None,
                 cor_method='pearson', linkage_method='complete',
                 optimal_ordering=True, key_added=None)

from matplotlib import rcParams
FIGSIZE = (15, 3)
rcParams["figure.figsize"] = FIGSIZE
sc.pl.dendrogram(adata, groupby='cell_type_final', save = '.svg')


In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, groupby=cluster_to_use, standard_scale="var", n_genes=5, dendrogram = True)

### Extract all cluster compared to all others in a single sheet


In [ ]:
sc.get.rank_genes_groups_df(adata, group=str("Astro TE"))

In [ ]:
### Extract all cluster compared to all others in a single sheet

dat = pd.DataFrame()
for i in adata.obs[cluster_to_use].unique():
    print(f"Cluster {i}")
    dat1 = sc.get.rank_genes_groups_df(adata, group=str(i))
    dat1['group'] = i
    dat = pd.concat([dat, dat1])

dat.to_csv(f"{dir_notebook}/analysis/{name_dir}/{name_dir}_{cluster_to_use}_markergenes.csv")

In [ ]:
### Compare two groups gene expression (whole section)
section_ = 'C3'
adata_temp = adata[adata.obs['section']==section_]
sc.tl.rank_genes_groups(adata_temp, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)
dat1 = sc.get.rank_genes_groups_df(adata_temp, group='APP')
dat1['group'] = 'APP'

dat1 = dat1[ ### Choose filters here
# (dat1['pct_nz_group'] > 0.15) & #Percentage of cell expressing the gene
(dat1['pvals_adj']<= 0.05) & # adjusted p-value
(abs(dat1['logfoldchanges']) > 0.26) # logfoldchange
]

dat1.to_csv(f"csv/{name_dir}/foldchange/{name_dir}_{section_}_whole_section_filter.csv")

## Subset (a)dataset for one cell type

In [ ]:
adata.obs['cell_type_final'].value_counts()

In [ ]:
adata2 = adata
celltype_to_subset = "SCH Gaba"
adata_microglia = adata2[adata2.obs['cell_type_final'] == celltype_to_subset]

In [ ]:
samples_ids_sub = adata_microglia.obs['sample'].unique()

In [ ]:
sc.pl.stacked_violin(adata_microglia, marker_genes, groupby='Genotype', dendrogram=False,)

In [ ]:
sc.pl.violin(adata_microglia, marker_genes, groupby='Genotype', order = ['WT','APP'],
             #log = True,
             # stripplot = False,
            )

In [ ]:
adata_microglia.obs['Genotype'].unique()

In [ ]:
sc.tl.rank_genes_groups(adata_microglia, groupby='run', method="wilcoxon", tie_correct = True, corr_method="benjamini-hochberg", pts = True)
dat1 = sc.get.rank_genes_groups_df(adata_microglia, group = 'circa4')

dat1.to_csv(f'{dir_notebook}/analysis/{name_dir}/{celltype_to_subset}_fold_change_no_threshold.csv', index=False)

In [ ]:
dat1 =  dat1.sort_values(by='logfoldchanges', ascending=False)
# dat1 = dat1[(dat1['pct_nz_group'] > 0.15)]
dat1

In [ ]:
dat1_filter = dat1[(~dat1['logfoldchanges'].between(-0.26,0.26))
                   & (dat1['pvals_adj'] <= 0.05)
                    & (dat1['pct_nz_group'] > 0.15)
                      ]
dat1_filter.reset_index(inplace=True)
dat1_filter.to_csv(f'{dir_notebook}/analysis/{name_dir}/{celltype_to_subset}_fold_change.csv', index=False)

### Volcano plot

In [ ]:
# dat1_filter = dat1[(~dat1['logfoldchanges'].between(-0.26,0.26)) & (dat1['pvals_adj'] <= 0.05) ]
# dat1_filter = dat1_filter.reset_index
# create sub pandas with genes over threshold
# plot after initial scatter with alpha = 1
# add text for each postion


plt.vlines(x=(-0.26,0.26), ymin=10e-24, ymax=1, color = "black", linestyles='dashed')
plt.hlines(y=0.05, xmin=-1, xmax=1, color = "black", linestyles='dashed')
plt.scatter(x=dat1['logfoldchanges'], y = dat1['pvals_adj'], alpha= 0.75, color = "grey", edgecolors=None)
plt.scatter(x= dat1_filter['logfoldchanges'], y=dat1_filter['pvals_adj'], alpha=1, color = 'red')
for idx, gene in enumerate(dat1_filter['names']):
    plt.text(dat1_filter['logfoldchanges'][idx], dat1_filter['pvals_adj'][idx], str(gene), color = 'black', fontsize = 12, ha= 'center')
plt.yscale('log')
plt.gca().invert_yaxis()
plt.xlabel('Log2 Foldchange')
plt.ylabel('Adjusted p-value')
plt.savefig('Gallery/volcano_plot_SCN.svg')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_microglia,
                                n_genes=5,
                                values_to_plot="logfoldchanges", cmap='bwr',
                                # vmin=-4,
                                # vmax=4,
                                )

In [ ]:
sc.pl.highest_expr_genes(adata_microglia, n_top=30, show=None, save=None, ax=None, gene_symbols=None, log=False)

In [ ]:
adata_microglia_WT=adata_microglia[adata_microglia.obs['Genotype']== 'WT']
adata_microglia_APP=adata_microglia[adata_microglia.obs['Genotype']== 'APP']

sc.pl.highest_expr_genes(adata_microglia_WT, n_top=30, show=None, save=None, ax=None, gene_symbols=None, log=False)
sc.pl.highest_expr_genes(adata_microglia_APP, n_top=30, show=None, save=None, ax=None, gene_symbols=None, log=False)

### Subcluster the subset

In [ ]:
# extract pca coordinates
X_pca = adata_microglia.obsm['X_pca'] 

sc.pp.pca(adata_microglia)
sc.pp.neighbors(adata_microglia)
sc.tl.umap(adata_microglia)

### Kmeans clustering
### You can choose the number of clusters by uncommenting n_clusters option
kmeans = KMeans(#n_clusters=2,
                random_state=0).fit(X_pca) 
adata_microglia.obs['kmeans'] = kmeans.labels_.astype(str)

sc.tl.leiden(adata_microglia, resolution = 0.2)

In [ ]:
### Choose one cluster to work with

# cluster_to_use = 'kmeans'
cluster_to_use = 'leiden'


In [ ]:
### Number of cells per clusters
max_clust = len(adata_microglia.obs[cluster_to_use].unique())
for i in range(0, max_clust):
    count = adata_microglia.obs[cluster_to_use].value_counts().iloc[i]
    print(f"Cluster {i} : {count} cells")

In [ ]:
### Generate a color palette for the clusters - to make color stay consistent across samples
adata_microglia.obs[cluster_to_use] = adata_microglia.obs[cluster_to_use].astype(str)

# Create a palette with a unique color for each cluster
num_clusters = len(adata_microglia.obs[cluster_to_use].astype(int).unique())
palette = sns.color_palette("hls", n_colors=num_clusters)

# Map each 'leiden' value to a color
adata_microglia.obs['leiden_colors'] = adata_microglia.obs[cluster_to_use].astype(int).apply(lambda x: palette[x])

In [ ]:
### Let's make UMAP plot. We will also add the cluster centroids to the plot
adata_microglia.obs['umap-1'] = adata_microglia.obsm['X_umap'][:, 0]
adata_microglia.obs['umap-2'] = adata_microglia.obsm['X_umap'][:, 1]
cluster_centroids = adata_microglia.obs.groupby(cluster_to_use)[['umap-1', 'umap-2']].median()

In [ ]:
from module.dataviz_analysis import umap_plot_indi_multi
samples_ids = samples_ids
umap_plot_indi_multi(adata_microglia,
                     cluster_to_use = "leiden",
                     individual_plot = True,
                     save_plot = False,
                     cmap_ = 'hls',
                     )

In [ ]:
samples_ids_sub

In [ ]:
# Map all cells
fig, axs = plt.subplots(3,4,figsize=(30, 15))
axs = axs.flatten()
clusters_plot = {0: 'magenta',1: 'cyan',2: 'green', "":'red', "":'orange',"7":'black',"":"purple"
                }

for idx, sample in enumerate(samples_ids_sub):
    adata_sel = adata_microglia[(adata_microglia.obs['sample'] == sample)]
    for cluster_id in adata_sel.obs[cluster_to_use].unique():
        cluster_data = adata_sel.obs[adata_sel.obs[cluster_to_use] == cluster_id]
        colors = clusters_plot[cluster_id] if cluster_id in clusters_plot else "none" ### for selected clusters in cluster_plot
        colors= cluster_data['leiden_colors'].unique()[0] ### for all clusters
        axs[idx].scatter(cluster_data['x_centroid'].astype('float'), cluster_data['y_centroid'].astype('float'), color=colors, s=10, label=cluster_id)
        axs[idx].set_title(f"Sample {sample}")
        # axs[idx].set_ylim(400,1300)
        # axs[idx].set_xlim(4100,5600)



In [ ]:
from module.dataviz_analysis import cluster_plot

cluster_plot(adata_microglia,
             cluster_to_use = 'cell_type_final',
             cluster_to_map = 'all',
             cmap_ = 'hls',
             save_plot = False,
            )

In [ ]:
### Correlation map of subclusters
cont_tab = pd.crosstab(adata_microglia.obs[cluster_to_use], adata_microglia.obs['mmc:supertype_name'], normalize="index")
plt.figure(figsize=(120, 10))
sns.heatmap(cont_tab, annot=True, cmap="YlGnBu", fmt=".1f")

In [ ]:
# Obtain cluster-specific differentially expressed genes
cluster_to_use = 'leiden'
# cluster_to_use = 'Genotype'
adata_microglia.obs[cluster_to_use] = adata_microglia.obs[cluster_to_use].astype(str)
# sc.tl.dendrogram(adata_microglia, groupby = cluster_to_use, n_pcs=None, use_rep=None, var_names=None, use_raw=None, cor_method='pearson', linkage_method='complete', optimal_ordering=False, key_added=None)
sc.tl.rank_genes_groups(adata_microglia, groupby=cluster_to_use, method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata_microglia, groupby=cluster_to_use, standard_scale="var", n_genes=5)

sc.pl.rank_genes_groups_dotplot(
    adata_microglia,
    n_genes=5,
    values_to_plot="logfoldchanges", cmap='bwr',
    vmin=-4,
    vmax=4,
)

In [ ]:
sc.pl.rank_genes_groups_violin(adata_microglia, groups=adata_microglia.obs[cluster_to_use], n_genes=1)

In [ ]:
ax = sc.pl.stacked_violin(adata_microglia, var_names = marker_genes , groupby='kmeans', dendrogram=True)

In [ ]:
# Obtain cluster-specific differentially expressed genes
# cluster_to_use = 'kmeans'
cluster_to_use = 'Genotype'
adata_microglia.obs[cluster_to_use] = adata_microglia.obs[cluster_to_use].astype(str)
# sc.tl.dendrogram(adata_microglia, groupby = cluster_to_use, n_pcs=None, use_rep=None, var_names=None, use_raw=None, cor_method='pearson', linkage_method='complete', optimal_ordering=False, key_added=None)
sc.tl.rank_genes_groups(adata_microglia, groupby=cluster_to_use, method="wilcoxon", tie_correct = True)
sc.pl.rank_genes_groups_dotplot(adata_microglia, groupby=cluster_to_use, standard_scale="var", n_genes=5)

sc.pl.rank_genes_groups_dotplot(
    adata_microglia,
    n_genes=5,
    values_to_plot="logfoldchanges", cmap='bwr',
    # vmin=-4,
    # vmax=4,
)

In [ ]:
### Extract gene expression per cluster + log fold change + p-value
cluster_to_use = 'Genotype'

adata_microglia.obs[cluster_to_use] = adata_microglia.obs[cluster_to_use].astype(str)
#sc.tl.dendrogram(adata, groupby = 'L04_newnum_subclassname')
sc.tl.rank_genes_groups(adata_microglia, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)

sc.pl.rank_genes_groups_dotplot(adata_microglia, groupby=cluster_to_use, standard_scale="var", n_genes=5)

dat = pd.DataFrame()
for i in adata2.obs[cluster_to_use].unique():
    print(f"Cluster {i}")
    dat1 = sc.get.rank_genes_groups_df(adata_microglia, group=i)
    dat1['group'] = i
    dat = pd.concat([dat, dat1])

### Extract not-normalized expression and clusters for individual cells
if not os.path.exists(f"{dir_notebook}/csv/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/csv/{name_dir}/")
dat.to_csv(f"{dir_notebook}/csv/{name_dir}/{name_dir}_clusters_foldchange_{celltype_to_subset}.csv")

In [ ]:
### TODO: update method of rank analysis (see "Output files" section)
dfs = []

a_ = 0
strerhyt = '-'
strenbis = ' '

all_celltype = adata_microglia.obs['leiden'].unique()

for cell_type_to_extract in all_celltype:
    adata_microglia_temp = adata_microglia[adata_microglia.obs['leiden'] == cell_type_to_extract]
    
    ### Extract gene expression per cluster + log fold change + p-value
    cluster_to_use = 'Genotype'
    clust_uniq = adata_microglia_temp.obs[cluster_to_use].unique()
    dat = pd.DataFrame()

    if (len(adata_microglia_temp[adata_microglia_temp.obs[cluster_to_use] == clust_uniq[0]]) < 2) or (len(adata_microglia_temp[adata_microglia_temp.obs[cluster_to_use] == clust_uniq[1]]) < 2):
        dat['nothing'] = ['to see here']
        dfs.append(dat)
        a_ +=1
        print(f'[{a_ * strerhyt}{(len(all_celltype) - a_)*strenbis}] Cluster {a_} / {len(all_celltype)}')
        continue

    adata_microglia_temp.obs[cluster_to_use] = adata_microglia_temp.obs[cluster_to_use].astype(str)
    #sc.tl.dendrogram(adata, groupby = 'L04_newnum_subclassname')
    sc.tl.rank_genes_groups(adata_microglia_temp, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)
    
    # sc.pl.rank_genes_groups_dotplot(adata_microglia, groupby=cluster_to_use, standard_scale="var", n_genes=5)
    
    for i in adata_microglia_temp.obs[cluster_to_use].unique():
        # print(f"Cluster {cell_type_to_extract}_{i}")
        dat1 = sc.get.rank_genes_groups_df(adata_microglia_temp, group=i)
        dat1['group'] = i
        dat = pd.concat([dat, dat1])

    dat = dat[ ### Choose filters here
    (dat['pct_nz_group'] > 0.15) & #Percentage of cell expressing the gene
    (dat['pvals_adj']<= 0.05) & # adjusted p-value
    (abs(dat['logfoldchanges']) > 0.26) # logfoldchange
    ]
    a_ +=1
    print(f'[{a_ * strerhyt}{(len(all_celltype) - a_)*strenbis}] Cluster {a_} / {len(all_celltype)}')

    dfs.append(dat)
else:
    print('Extraction done')


import xlsxwriter
writer = pd.ExcelWriter(f'csv/{name_dir}/{celltype_to_subset}_subclusters_df_comb.xlsx', engine='xlsxwriter')
for j in range(0,len(dfs)):
    dfs[j].to_excel(writer, sheet_name=all_celltype[j], index=False)

writer.close()

## Subset one region

In [ ]:
adata2.obs['region_automap'].unique()

In [ ]:
region_to_subset = "SCH"
adata_region = adata2[adata2.obs['region_automap'] == region_to_subset]

In [ ]:
adata_region.obs[adata_region.obs['cell type']== 'HY GABA'].groupby('mmc:subclass_name')['cell_id'].nunique()

In [ ]:
adata_region.obs[adata_region.obs['cell type']=='Astro']['cell_type_newnum'].head()

In [ ]:
# Generate new numbering base on unique 'cell type'
all_cell_type = adata_region.obs['cell type'].unique()
list_cell_nb = range(0, len(all_cell_type))
mapping_dict = dict(zip(all_cell_type,list_cell_nb))
adata_region.obs['cell_type_newnum'] = adata_region.obs['cell type'].map(mapping_dict)
# mapping_dict

### Generate a color palette for the clusters - to make color stay consistent across samples
adata_region.obs['cell_type_newnum'] = adata_region.obs['cell_type_newnum'].astype(int)

# Create a palette with a unique color for each cluster
num_clusters = len(adata_region.obs['cell_type_newnum'].unique())
palette = sns.color_palette("bright", n_colors=num_clusters)

# Map each 'leiden' value to a color
adata_region.obs['kmeans_colors'] = adata_region.obs['cell_type_newnum'].apply(lambda x: palette[x])

# Mapping of clusters
fig, axs = plt.subplots(3,2,figsize=(15, 18))
axs = axs.flatten()
clusters_plot = { 1: 'lightcoral', 11:'black',4:'red',
    # 0: 'orchid', 1: 'forestgreen',2: 'coral', 4:'orange',
    # 3:'red', 5:'blue',6:'cyan',7:'black'
    # 4:'red',0:'black'
}

for idx, sample in enumerate(samples_ids):
    adata_sel = adata_region[(adata_region.obs['sample'] == sample)]
    for cluster_id in adata_sel.obs['cell_type_newnum'].unique():
        cluster_data = adata_sel.obs[adata_sel.obs['cell_type_newnum'] == cluster_id]
        if len(cluster_data) >= 0:
            colors = clusters_plot[cluster_id] if cluster_id in clusters_plot else "none"
            colors= cluster_data['kmeans_colors'].unique()[0]
            axs[idx].scatter(cluster_data['x_centroid'], cluster_data['y_centroid'], color=colors, s=15, label=cluster_data['cell type'].unique()[0])
            axs[idx].set_title(f"Sample {sample}")

plt.legend(markerscale=1, scatterpoints=1000, bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)


In [ ]:
# len(adata_region)/1000
# adata_region_notunique.obs.groupby('cell type')['cell_id'].nunique()
# list_to_exclude

In [ ]:
list_to_exclude  = adata_region.obs.groupby('cell type')['cell_id'].nunique() >= 10
list_to_exclude.values
dict_exclude = dict(zip(list_to_exclude.index, list_to_exclude.values))
dict_exclude

adata_region.obs['exclude'] = adata_region.obs['cell type'].map(dict_exclude)

adata_region_notunique = adata_region[adata_region.obs['exclude'] != False]

adata_region_notunique.obs.groupby('cell type')['cell_id'].nunique()

sc.tl.dendrogram(adata_region_notunique, groupby = 'cell type', n_pcs=None, use_rep=None, var_names=None, use_raw=None, cor_method='pearson', linkage_method='complete', optimal_ordering=False, key_added=None)
sc.tl.rank_genes_groups(adata_region_notunique, groupby='cell type', method="wilcoxon", tie_correct = True)
sc.pl.rank_genes_groups_dotplot(adata_region_notunique, groupby='cell type', standard_scale="var", n_genes=5)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata_region_notunique,
    n_genes=5,
    values_to_plot="logfoldchanges", cmap='bwr',
    # vmin=-4,
    # vmax=4,
)

In [ ]:
### Extract gene expression per cluster + log fold change + p-value
cluster_to_use = 'Genotype'

adata_region.obs[cluster_to_use] = adata_region.obs[cluster_to_use].astype(str)
#sc.tl.dendrogram(adata, groupby = 'L04_newnum_subclassname')
sc.tl.rank_genes_groups(adata_region, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)

sc.pl.rank_genes_groups_dotplot(adata_region, groupby=cluster_to_use, standard_scale="var", n_genes=5)

dat = pd.DataFrame()
for i in adata_region.obs[cluster_to_use].unique():
    print(f"Cluster {i}")
    dat1 = sc.get.rank_genes_groups_df(adata_region, group=i)
    dat1['group'] = i
    dat = pd.concat([dat, dat1])

### Extract not-normalized expression and clusters for individual cells
if not os.path.exists(f"{dir_notebook}/csv/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/csv/{name_dir}/")
dat.to_csv(f"{dir_notebook}/csv/{name_dir}/{name_dir}_clusters_foldchange_{region_to_subset}-region.csv")

In [ ]:
celltype_to_subset = "Astro"
adata_region_cell = adata_region[adata_region.obs['cell type'] == celltype_to_subset]

In [ ]:
### Extract gene expression per cluster + log fold change + p-value
cluster_to_use = 'Genotype'

adata_region_cell.obs[cluster_to_use] = adata_region_cell.obs[cluster_to_use].astype(str)
#sc.tl.dendrogram(adata, groupby = 'L04_newnum_subclassname')
sc.tl.rank_genes_groups(adata_region_cell, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)

sc.pl.rank_genes_groups_dotplot(adata_region_cell, groupby=cluster_to_use, standard_scale="var", n_genes=5)

dat = pd.DataFrame()
for i in adata_region_cell.obs[cluster_to_use].unique():
    print(f"Cluster {i}")
    dat1 = sc.get.rank_genes_groups_df(adata_region_cell, group=i)
    dat1['group'] = i
    dat = pd.concat([dat, dat1])

### Extract not-normalized expression and clusters for individual cells
if not os.path.exists(f"{dir_notebook}/csv/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/csv/{name_dir}/")
dat.to_csv(f"{dir_notebook}/csv/{name_dir}/{name_dir}_clusters_foldchange_{region_to_subset}-region-{celltype_to_subset}-cell.csv")

# Output files

## Extract foldchange, pvalue, etc.

In [ ]:
adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz")

In [ ]:
gene_names = np.array(adata.var_names)
gene_names = sorted(gene_names)
# print(gene_names)

## DEG whole brain

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='run', method="wilcoxon", tie_correct = True, pts = True)

In [ ]:
filters_dic = {'percentage_pop': 0.10,
               'pval_adj' : 0.05,
               'logfoldchanges' : 0.26,
               'mean_count' : 0.01}

for i in adata.obs['run'].unique():
    # print(f"Cluster {cell_type_to_extract}_{i}")
    dat = sc.get.rank_genes_groups_df(adata, group="SD1")
    # sc.pp.calculate_qc_metrics(adata_sub, inplace=True)
    dat["mean_count"] = dat["names"].map(dict(zip(adata.var.index, adata.var.mean_counts)))

dat_filter = dat[ ### Choose filters here
        # (dat['pct_nz_group'] > filters_dic['percentage_pop']) & #Percentage of cell expressing the gene
        (dat['pvals_adj']<= filters_dic['pval_adj']) & # adjusted p-value
        (abs(dat['logfoldchanges']) > filters_dic['logfoldchanges']) & # logfoldchange
        (dat['mean_count'] >= filters_dic['mean_count'])
        ]


In [ ]:
dat.shape,dat_filter.shape

In [ ]:
# ### Extract normalized expression and clusters for individual cells
if not os.path.exists(f"{dir_notebook}/analysis/{name_dir}/foldchanges/wholebrain"):
   os.makedirs(f"{dir_notebook}/analysis/{name_dir}/foldchanges/wholebrain")
dat.to_csv(f"{dir_notebook}/analysis/{name_dir}/foldchanges/wholebrain/{name_dir}_foldchange_wholesection.csv")
dat_filter.to_csv(f"{dir_notebook}/analysis/{name_dir}/foldchanges/wholebrain/{name_dir}_foldchange_wholesection_filter.csv")

## DEG per celltypes

In [ ]:
all_celltype = np.array(adata.obs['cell_type_final'].unique())
all_cell_type_sheet = all_celltype

In [ ]:
import progressbar

dfs = []
dfs_filter = []

filters_ = True
filters_dic = {'percentage_pop': 0.10,
               'pval_adj' : 0.05,
               'logfoldchanges' : 0.26,
               'mean_count' : 0.01}

a_ = 0
bar = progressbar.ProgressBar(maxval=len(all_celltype)+1, \
    widgets=[progressbar.Bar('=', '[', ']'), ' ', progressbar.Percentage()])
bar.start()

for cell_type_to_extract in all_celltype:
    adata_microglia = adata[adata.obs['cell_type_final'] == cell_type_to_extract]
    sc.pp.calculate_qc_metrics(adata_microglia, inplace=True)

    print(f"Start analysis of {cell_type_to_extract}")
    ### Extract gene expression per cluster + log fold change + p-value
    cluster_to_use = 'run'
    clust_uniq = adata.obs[cluster_to_use].unique()
    dat = pd.DataFrame()

    if (len(adata_microglia[adata_microglia.obs[cluster_to_use] == clust_uniq[0]]) < 2) or (len(adata_microglia[adata_microglia.obs[cluster_to_use] == clust_uniq[1]]) < 2):
        all_cell_type_sheet = np.delete(all_cell_type_sheet, np.where(all_cell_type_sheet == cell_type_to_extract))
        # a_ +=1
        continue

    adata_microglia.obs[cluster_to_use] = adata_microglia.obs[cluster_to_use].astype(str)
    #sc.tl.dendrogram(adata, groupby = 'L04_newnum_subclassname')
    sc.tl.rank_genes_groups(adata_microglia, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)
    
    # sc.pl.rank_genes_groups_dotplot(adata_microglia, groupby=cluster_to_use, standard_scale="var", n_genes=5)
    
    for i in adata.obs[cluster_to_use].unique():
        # print(f"Cluster {cell_type_to_extract}_{i}")
        dat1 = sc.get.rank_genes_groups_df(adata_microglia, group=i)
        dat1['group'] = i
        dat = pd.concat([dat, dat1])
        dat["mean_count"] = dat["names"].map(dict(zip(adata_microglia.var.index, adata_microglia.var.mean_counts)))

    if filters_ == True:
        dat_filter = dat[ ### Choose filters here
        # (dat['pct_nz_group'] > filters_dic['percentage_pop']) & #Percentage of cell expressing the gene
        (dat['pvals_adj']<= filters_dic['pval_adj']) & # adjusted p-value
        (abs(dat['logfoldchanges']) > filters_dic['logfoldchanges']) & # logfoldchange
        (dat['mean_count'] >= filters_dic['mean_count'])
        ]
        dfs_filter.append(dat_filter)

    a_ +=1
    clear_output()
    bar.update(a_+1)

    dfs.append(dat)
else:
    clear_output()
    print('Extraction done')

bar.finish()

In [ ]:
if not os.path.exists(f"{dir_notebook}/analysis/{name_dir}/foldchanges/celltype"):
   os.makedirs(f"{dir_notebook}/analysis/{name_dir}/foldchanges/celltype")

writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/foldchanges/celltype/DEG_celltype_no-filter.xlsx', engine='xlsxwriter')
for j in range(0,len(dfs)):
    dfs[j].to_excel(writer, sheet_name=all_cell_type_sheet[j], index=False)
writer.close()

writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/foldchanges/celltype/DEG_celltype_filter.xlsx', engine='xlsxwriter')
for j in range(0,len(dfs)):
    dfs_filter[j].to_excel(writer, sheet_name=all_cell_type_sheet[j], index=False)
writer.close()

## DEG per regions

In [ ]:
adata.obs['region_automap_name'] =adata.obs['region_automap_name'].cat.remove_categories(np.nan)

In [ ]:
all_regions = adata.obs['region_automap_name'].unique()
all_regions_sheet = list(all_regions)
all_regions_sheet

In [ ]:
import progressbar

dfs = []
dfs_filter = []

filters_ = True
filters_dic = {'percentage_pop': 0.10,
               'pval_adj' : 0.05,
               'logfoldchanges' : 0.26,
               'mean_count' : 0.01}

a_ = 0
bar = progressbar.ProgressBar(maxval=len(all_regions)+1, \
    widgets=[progressbar.Bar('=', '[', ']'), ' ', progressbar.Percentage()])
bar.start()


for region_to_extract in all_regions:
    print(f"Start processing : {region_to_extract}")
    adata_microglia = adata[adata.obs['region_automap_name'] == region_to_extract]
    sc.pp.calculate_qc_metrics(adata_microglia, inplace=True)

    ### Extract gene expression per cluster + log fold change + p-value
    cluster_to_use = 'run'
    clust_uniq = adata.obs[cluster_to_use].unique()
    dat = pd.DataFrame()

    if (len(adata_microglia[adata_microglia.obs[cluster_to_use] == clust_uniq[0]]) < 2) or (len(adata_microglia[adata_microglia.obs[cluster_to_use] == clust_uniq[1]]) < 2):
        all_regions_sheet = np.delete(all_regions, np.where(all_regions == region_to_extract))
        # a_ +=1
        continue

    adata_microglia.obs[cluster_to_use] = adata_microglia.obs[cluster_to_use].astype(str)
    sc.tl.rank_genes_groups(adata_microglia, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)
    
   
    for i in adata.obs[cluster_to_use].unique():
        print(f"Cluster {i}")
        dat1 = sc.get.rank_genes_groups_df(adata_microglia, group=i)
        dat1['group'] = i
        dat = pd.concat([dat, dat1])
        dat["mean_count"] = dat["names"].map(dict(zip(adata_microglia.var.index, adata_microglia.var.mean_counts)))
        

    if filters_ == True:
        dat_filter = dat[ ### Choose filters here
        # (dat['pct_nz_group'] > filters_dic['percentage_pop']) & #Percentage of cell expressing the gene
        (dat['pvals_adj']<= filters_dic['pval_adj']) & # adjusted p-value
        (abs(dat['logfoldchanges']) > filters_dic['logfoldchanges']) & # logfoldchange
        (dat['mean_count'] >= filters_dic['mean_count'])
        ]
        dfs_filter.append(dat_filter)

    a_ +=1
    clear_output()
    bar.update(a_+1)

    dfs.append(dat)
else:
    clear_output()
    print('Extraction done')



In [ ]:
if not os.path.exists(f"{dir_notebook}/analysis/{name_dir}/foldchanges/region"):
   os.makedirs(f"{dir_notebook}/analysis/{name_dir}/foldchanges/region")

writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/foldchanges/region/DEG_region_no-filter.xlsx', engine='xlsxwriter')
for j in range(0,len(dfs)):
    dfs[j].to_excel(writer, sheet_name=all_regions_sheet[j], index=False)
writer.close()

writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/foldchanges/region/DEG_region_filter.xlsx', engine='xlsxwriter')
for j in range(0,len(dfs)):
    dfs_filter[j].to_excel(writer, sheet_name=all_regions_sheet[j], index=False)
writer.close()

### DEG celltype comparaison in different regions (single run)

In [ ]:
run_to_analyze = 'circa4'

adata_sub = adata[adata.obs['run'] == run_to_analyze]

# all_cell_type = np.array(adata_sub.obs['cell_type_final'].unique())
all_cell_type = np.array(['Astro TE','Oligodendrocyte','OPC',"Microglia","Endothelial"])
all_cell_type_sheet = all_cell_type

In [ ]:
import progressbar

dfs = []
dfs_filter = []

filters_ = True
filters_dic = {'percentage_pop': 0.10,
               'pval_adj' : 0.05,
               'logfoldchanges' : 0.26,
               'mean_count' : 0.01}

a_ = 0
bar = progressbar.ProgressBar(maxval=len(all_cell_type)+1, \
    widgets=[progressbar.Bar('=', '[', ']'), ' ', progressbar.Percentage()])
bar.start()


for cell in all_cell_type:
    print(cell)
    adata_microglia = adata_sub[adata_sub.obs['cell_type_final'] == cell]
    sc.pp.calculate_qc_metrics(adata_microglia, inplace=True)


    if adata_microglia.shape[0] == 0:
        print(f'Subset adata for {cell} empty. Check parameters')
        break
    
    adata_microglia = adata_microglia[~adata_microglia.obs['region_automap_name'].isin(['VLMC',"Ependymal"])]

    ### Extract gene expression per cluster + log fold change + p-value
    cluster_to_use = 'region_automap_name'
    clust_uniq = adata_sub.obs[cluster_to_use].unique()
    dat = pd.DataFrame()

    if (len(adata_microglia[adata_microglia.obs[cluster_to_use] == clust_uniq[0]]) < 2) or (len(adata_microglia[adata_microglia.obs[cluster_to_use] == clust_uniq[1]]) < 2):
        all_cell_type_sheet = np.delete(all_cell_type_sheet, np.where(all_cell_type_sheet==cell))
        # a_ +=1
        continue

    # adata_microglia.obs[cluster_to_use] = adata_microglia.obs[cluster_to_use].astype(str)
    sc.tl.rank_genes_groups(adata_microglia, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)
    
   
    for i in adata_microglia.obs[cluster_to_use].unique():
        # print(f"Cluster {cell_type_to_extract}_{i}")
        dat1 = sc.get.rank_genes_groups_df(adata_microglia, group=i)
        dat1['group'] = i
        dat = pd.concat([dat, dat1])
        dat["mean_count"] = dat["names"].map(dict(zip(adata_microglia.var.index, adata_microglia.var.mean_counts)))


    if filters_ == True:
        dat_filter = dat[ ### Choose filters here
        # (dat['pct_nz_group'] > filters_dic['percentage_pop']) & #Percentage of cell expressing the gene
        (dat['pvals_adj']<= filters_dic['pval_adj']) & # adjusted p-value
        (abs(dat['logfoldchanges']) > filters_dic['logfoldchanges']) & # logfoldchange
        (dat['mean_count'] >= filters_dic['mean_count'])
        ]
        dfs_filter.append(dat_filter)

    a_ +=1
    clear_output()

    bar.update(a_+1)

    dfs.append(dat)
else:
    print('Extraction done')



In [ ]:
if not os.path.exists(f"{dir_notebook}/analysis/{name_dir}/DEG_celltype_per_region/{run_to_analyze}"):
   os.makedirs(f"{dir_notebook}/analysis/{name_dir}/DEG_celltype_per_region/{run_to_analyze}")

import xlsxwriter
writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/DEG_celltype_per_region/{run_to_analyze}/{run_to_analyze}_DEG_celltype_per_region_no-filter.xlsx', engine='xlsxwriter')
for j in range(0,len(dfs)):
    dfs[j].to_excel(writer, sheet_name=list(all_cell_type)[j], index=False)
writer.close()

if filters_ == True:
    writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/DEG_celltype_per_region/{run_to_analyze}/{run_to_analyze}_DEG_celltype_per_region_filter.xlsx', engine='xlsxwriter')
    for j in range(0,len(dfs)):
        dfs_filter[j].to_excel(writer, sheet_name=list(all_cell_type)[j], index=False)
    writer.close()

## DEG per celltypes and regions

In [ ]:
all_regions = np.array(adata.obs['region_automap_name'].unique())
all_regions

In [ ]:
all_celltype = np.array(adata.obs['cell_type_final'].unique())
all_celltype

In [ ]:
filters_ = True
filters_dic = {'percentage_pop': 0.10,
               'pval_adj' : 0.05,
               'logfoldchanges' : 0.26,
               'mean_count' : 0.01}

a_ = 0
bar = progressbar.ProgressBar(maxval=len(all_regions)+1, \
    widgets=[progressbar.Bar('=', '[', ']'), ' ', progressbar.Percentage()])
bar.start()



for region_to_extract in all_regions:
    
    cluster_to_use = 'run'
    clust_uniq = adata.obs[cluster_to_use].unique()
    
    adata3 = adata[adata.obs['region_automap_name'] == region_to_extract]

    
    if (len(adata3[adata3.obs[cluster_to_use] == clust_uniq[0]]) < 2) or (len(adata3[adata3.obs[cluster_to_use] == clust_uniq[1]]) < 2):
        continue

    dfs = []
    dfs_filter = []

    all_celltype = np.array(adata3.obs['cell_type_final'].unique())
    all_celltype_sheet = all_celltype
    
    
    for cell_type_to_extract in all_celltype:
        print(f'Starting region : {region_to_extract}')
        bar.update(a_)
        print(f'Celltype : {cell_type_to_extract}')

        adata_microglia = adata3[adata3.obs['cell_type_final'] == cell_type_to_extract]
        sc.pp.calculate_qc_metrics(adata_microglia, inplace=True)

        ### Extract gene expression per cluster + log fold change + p-value
        
        dat = pd.DataFrame()

        if (len(adata_microglia[adata_microglia.obs[cluster_to_use] == clust_uniq[0]]) < 2) or (len(adata_microglia[adata_microglia.obs[cluster_to_use] == clust_uniq[1]]) < 2):
            all_celltype_sheet = np.delete(all_celltype_sheet, np.where(all_celltype_sheet==cell_type_to_extract))
            clear_output()
            continue

        adata_microglia.obs[cluster_to_use] = adata_microglia.obs[cluster_to_use].astype(str)
        sc.tl.rank_genes_groups(adata_microglia, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)
        
        for i in adata3.obs[cluster_to_use].unique():
            # print(f"Cluster {cell_type_to_extract}_{i}")
            dat1 = sc.get.rank_genes_groups_df(adata_microglia, group=i)
            dat1['group'] = i
            dat = pd.concat([dat, dat1])
            dat["mean_count"] = dat["names"].map(dict(zip(adata_microglia.var.index, adata_microglia.var.mean_counts)))


        if filters_ == True:
            dat_filter = dat[ ### Choose filters here
                # (dat['pct_nz_group'] > filters_dic['percentage_pop']) & #Percentage of cell expressing the gene
                (dat['pvals_adj']<= filters_dic['pval_adj']) & # adjusted p-value
                (abs(dat['logfoldchanges']) > filters_dic['logfoldchanges']) & # logfoldchange
                (dat['mean_count'] >= filters_dic['mean_count'])
            ]
            dfs_filter.append(dat)

        dfs.append(dat)
        clear_output()
        
    if not os.path.exists(f"{dir_notebook}/analysis/{name_dir}/foldchanges/celltype_in_region"):
       os.makedirs(f"{dir_notebook}/analysis/{name_dir}/foldchanges/celltype_in_region")

    writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/foldchanges/celltype_in_region/{region_to_extract}_all_celltypes_DEG.xlsx', engine='xlsxwriter')
    for j in range(0,len(dfs)):
        dfs[j].to_excel(writer, sheet_name=all_celltype_sheet[j], index=False)
    writer.close()

    writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/foldchanges/celltype_in_region/{region_to_extract}_all_celltypes_DEG_filter.xlsx', engine='xlsxwriter')
    for j in range(0,len(dfs_filter)):
        dfs_filter[j].to_excel(writer, sheet_name=all_celltype_sheet[j], index=False)
    writer.close()

    a_ +=1
    clear_output()

    bar.update(a_)
    
else:
    clear_output()
    print('Extraction done')

## DEG : Region comparison per celltype

In [ ]:
all_regions = np.array(adata.obs['region_automap_name'].unique())
all_celltype = np.array(adata.obs['cell_type_final'].unique())
all_region_sheet = all_regions

In [ ]:
filters_ = True

for cell_to_extract in all_celltype:
    
    cluster_to_use = 'run'
    clust_uniq = adata.obs[cluster_to_use].unique()
    
    adata3 = adata[adata.obs['cell_type_final'] == cell_to_extract]

    
    if (len(adata3[adata3.obs[cluster_to_use] == clust_uniq[0]]) < 2) or (len(adata3[adata3.obs[cluster_to_use] == clust_uniq[1]]) < 2):
        continue

    dfs = []
    dfs_filter = []

    all_regions = np.array(adata3.obs['region_automap_name'].unique())
    print(f'Starting celltype : {cell_to_extract}')

    for region in all_regions:
        adata_microglia = adata3[adata3.obs['region_automap_name'] == region]
        sc.pp.calculate_qc_metrics(adata_microglia, inplace=True)
        ### Extract gene expression per cluster + log fold change + p-value
        
        
        dat = pd.DataFrame()

        if (len(adata_microglia[adata_microglia.obs[cluster_to_use] == clust_uniq[0]]) < 2) or (len(adata_microglia[adata_microglia.obs[cluster_to_use] == clust_uniq[1]]) < 2):
            all_region_sheet = np.delete(all_region_sheet, np.where(all_region_sheet==region))
            continue

        adata_microglia.obs[cluster_to_use] = adata_microglia.obs[cluster_to_use].astype(str)
        sc.tl.rank_genes_groups(adata_microglia, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)
        
        for i in adata3.obs[cluster_to_use].unique():
            # print(f"Cluster {region}_{i}")
            dat1 = sc.get.rank_genes_groups_df(adata_microglia, group=i)
            dat1['group'] = i
            dat = pd.concat([dat, dat1])
            dat["mean_count"] = dat["names"].map(dict(zip(adata_microglia.var.index, adata_microglia.var.mean_counts)))

        if filters_ == True:
            dat_filter = dat[ ### Choose filters here
            (dat['pct_nz_group'] > 0.15) & #Percentage of cell expressing the gene
            (dat['pvals_adj']<= 0.05) & # adjusted p-value
            (abs(dat['logfoldchanges']) > 0.26) & # logfoldchange
            (dat['mean_count'] >= filters_dic['mean_count'])
            ]
            dfs_filter.append(dat)

        dfs.append(dat)
    else:
        print('Extraction done')


    
    writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/{cell_to_extract}_all_region_DEG.xlsx', engine='xlsxwriter')
    for j in range(0,len(dfs)):
        dfs[j].to_excel(writer, sheet_name=all_region_sheet[j], index=False)
    writer.close()

    writer = pd.ExcelWriter(f'{dir_notebook}/analysis/{name_dir}/{cell_to_extract}_all_region_DEG_filter.xlsx', engine='xlsxwriter')
    for j in range(0,len(dfs_filter)):
        dfs_filter[j].to_excel(writer, sheet_name=all_region_sheet[j], index=False)
    writer.close()

## Normalized gene counts with all annotations

In [ ]:
df = pd.read_parquet(f'{dir_notebook}/csv/{name_dir}/{name_dir}_norm_combined.parquet')

In [ ]:
df = pd.DataFrame(data=adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)

from module.xenium_preprocessing import add_annotations
df = add_annotations(adata,df)
df.to_parquet(f"{dir_notebook}/csv/{name_dir}/{name_dir}_norm_combined.parquet")

In [ ]:
df[(df['run']=='circa4')&(df['cell_type_final']=="SCH Gaba")]['Bbc3'].value_counts()

In [ ]:
### Not optimized. Need to refactor with .groupby

from module.misc import genes_list

celltype_ = adata.obs['cell_type_final'].unique()



marker_genes = genes_list('panel_5k')

gene_dfs = []
genes_df = pd.DataFrame()
sample_list = pd.DataFrame(adata.obs['sample'].unique())

for celltype in celltype_:
    df2 = df[df['cell_type_final'] == celltype]
    genes_df = pd.DataFrame()
    for gene_ in marker_genes:
        temp = df2.groupby('sample')[gene_].mean()
        temp_df = pd.DataFrame(temp)
        genes_df = pd.concat([genes_df, temp_df], axis = 1)
    gene_dfs.append(genes_df)


import xlsxwriter
writer = pd.ExcelWriter(f'csv/{name_dir}/{name_dir}_gene_dfs.xlsx', engine='xlsxwriter')
for j in range(0,len(gene_dfs)):
    gene_dfs[j].to_excel(writer, sheet_name=celltype_[j], index=False)

writer.close()

## Extract score, FC, p-value for each genes, compared to other clusters

In [ ]:
### Extract gene expression per cluster + log fold change + p-value
import progressbar
cluster_to_use = 'sample'
cluster_to_use = 'run'
# cluster_to_use = 'Genotype'

all_celltype = np.array(adata.obs['cell_type_final'].unique())

# adata2.obs[cluster_to_use] = adata2.obs[cluster_to_use].astype(str)
# #sc.tl.dendrogram(adata, groupby = 'L04_newnum_subclassname')
# sc.tl.rank_genes_groups(adata2, groupby=cluster_to_use, method="wilcoxon", tie_correct = True)
# sc.pl.rank_genes_groups_dotplot(adata2, groupby=cluster_to_use, standard_scale="var", n_genes=5)

a_ = 0
bar = progressbar.ProgressBar(maxval=len(all_celltype), \
    widgets=[progressbar.Bar('=', '[', ']'), ' ', progressbar.Percentage()])
bar.start()

dat = pd.DataFrame()
for i in adata2.obs[cluster_to_use].unique():
    print(f"Cluster {i}")
    dat1 = sc.get.rank_genes_groups_df(adata2, group=str(i))
    dat1['group'] = i
    dat = pd.concat([dat, dat1])
    a_ +=1
    bar.update(a_+1)

bar.finish()

### Extract not-normalized expression and clusters for individual cells
if not os.path.exists(f"{dir_notebook}/analysis/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/analysis/{name_dir}/")
dat.to_csv(f"{dir_notebook}/analysis/{name_dir}/{name_dir}_clusters_foldchange_allbrain.csv")

In [ ]:
dat[dat['group'] == "SD1"].sort_values(by='logfoldchanges')

## Extract cell population comparison

In [ ]:
adata_region = adata[adata.obs['region_automap_name'] == 'SCH']

In [ ]:
# adata_region.obs.groupby('Genotype', as_index=False)['cell_type_final'].value_counts(normalize=False)
df_t = adata_region.obs.groupby('sample', as_index=False)[['Genotype','cell_type_final']].value_counts(normalize=False)
adata_region.obs.value_counts(['Genotype','cell_type_final',], normalize=False)


In [ ]:
df_t.to_csv(f'csv/{name_dir}/{name_dir}_SCH_cellpop.csv')

## Extract cell population

In [ ]:
adata.obs.groupby('run')['cell_type_final'].value_counts().to_csv(f'{dir_notebook}/analysis/{name_dir}/summary_cell_number.csv')

In [ ]:
cell_nb = pd.read_csv(f'{dir_notebook}/analysis/{name_dir}/summary_cell_number.csv')
cell_nb_SD = cell_nb[cell_nb['run'] == 'SD1']
cell_nb_circa = cell_nb[cell_nb['run'] == 'circa4']
cell_nb_circa.head(1)

## Extract cell pop per region

In [ ]:
grouped = adata.obs.groupby(['run','sample','region_automap_name'])['cell_type_final'].value_counts(dropna=True)
grouped.to_csv('data/test_cell_count.csv')
df = pd.read_csv('data/test_cell_count.csv')

## Gene expression

In [ ]:
run_ = 'circa4'
df_sub = df[df['run'] == run_]

from module.misc import genes_list

gene_list = genes_list('panel_5k')



In [ ]:
for region in df_sub['region_automap_name'].unique():
    ddf_sub = df_sub[df_sub['region_automap_name'] == region]
    ddf = ddf_sub.groupby('cell_type_final')[gene_list].mean()
    ddf = ddf.T

    path_to_save = f"../R/data/gene_list_{run_}/{region}"
    if not os.path.exists(path_to_save):
        os.makedirs(path_to_save)

    for cell in ddf.keys():
        ddf_temp = dict(zip(ddf[cell].index, ddf[cell].values))
        ddf_temp = {key:value for key, value in ddf_temp.items() if value > 0.01}
        gene_expressed = [key for key in ddf_temp.keys()]
        gene_expressed = pd.Series(gene_expressed)
        gene_expressed.to_csv(f"{path_to_save}/{cell}.csv")

In [ ]:
import os
run_ = 'circa4'
directory_expressed_genes = f"../R/data/gene_list_{run_}"
all_files = os.listdir(directory_expressed_genes)
all_files = [file for file in all_files if file.split('.')[-1]=="csv"]

cells = []
expressed_genes = [] 

for filename in all_files:
    celltype = filename.split('.')[0]
    celllist = pd.read_csv(f'{directory_expressed_genes}/{filename}')
    cells.append(celltype)   
    expressed_genes.append(len(celllist))

len(cells),len(expressed_genes)

df_genlist = pd.DataFrame(data = {"Celltype" : cells,
                                  "Count" : expressed_genes})

df_genlist.to_csv('../notebook/analysis/circa-SD/NS-genexpression-summary.csv')

# Test

## Spaco (opti color)

In [ ]:
import spaco

In [ ]:
adata2

In [ ]:
adata2 = adata[adata.obs['sample']=="3159-1"].copy()
adata2.obsm['spatial']=adata2.obsm['coord_xy'].copy()

In [ ]:
sc.set_figure_params(figsize=(10,15), facecolor="white", dpi_save=300)
sc.pl.spatial(adata2, color="region_automap_name", spot_size=15, palette = "tab20")

palette_default = adata2.uns['region_automap_name_colors'].copy()
sns.palplot(palette_default)

In [ ]:
# Get optimized color-cluster assignment with Spaco
color_mapping = spaco.colorize(
    cell_coordinates=adata2.obsm['spatial'],
    cell_labels=adata2.obs['region_automap_name'],
    colorblind_type="none",
    radius=0.05, # radius is related to the physical scaling of .obsm['spatial']
    n_neighbors=30,
    palette=palette_default, # if `palette` is specified, the `colorize` function only refines the assignment.
)

color_mapping

In [ ]:
# Get visualization with optimized color assignment
color_mapping = {k: color_mapping[k] for k in adata2.obs['region_automap_name'].cat.categories}

# Set new colors for adata
palette_spaco = list(color_mapping.values())

# Spaco colorization
sc.pl.spatial(adata2, color="region_automap_name", spot_size=15, palette=palette_spaco)
sns.palplot(palette_spaco)

## t-SNE


In [ ]:
sc.tl.tsne(adata, use_rep="X_pca")

In [ ]:
sc.pl.tsne(adata, color=["leiden"], cmap="tab20b")

In [ ]:
sc.pl.tsne(adata, color=["cell_type_final"], cmap="tab20b", size=0.5)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=50)
sc.tl.leiden(adata, n_iterations=2, directed=False, key_added = "leidentsne", resolution=2)

In [ ]:
sc.pl.tsne(adata, color=["leidentsne"], cmap="tab20b", size = 0.5)

In [ ]:
### Correlation map
cont_tab = pd.crosstab(adata.obs['leidentsne'], adata.obs['mmc:subclass_name'], normalize="index")
cont_tab = cont_tab.loc[:, cont_tab.sum(axis=0) > 0.1] 
# cont_tab.sort_values(by='149 PVT-PT Ntrk1 Glut',ascending=False, inplace=True)
plt.figure(figsize=(20,20))
sns.heatmap(cont_tab.T, annot=True, cmap="YlGnBu", fmt=".1f", cbar = False)

In [ ]:
clustering_method = "leidentsne"

### Generate a color palette for the clusters - to make color stay consistent across samples
adata.obs[clustering_method] = adata.obs[clustering_method].astype(int)

# Create a palette with a unique color for each cluster
num_clusters = len(adata.obs[clustering_method].unique())
palette = sns.color_palette("tab20", n_colors=num_clusters)

# Map each 'leiden' value to a color
adata.obs['kmeans_colors'] = adata.obs[clustering_method].apply(lambda x: palette[x])

# Mapping of clusters
fig, axs = plt.subplots(4,3,figsize=(15, 20))
axs = axs.flatten()
clusters_plot = {
    0: 'orchid', 1: 'forestgreen', 2: 'black', 3:'red', 4:'cyan', 5:'blue', 6:'darkorange',7:'coral',
    # 8:'forestgreen', 9: 'coral',10:'red', 11:'cyan',
    # 6:'blue',0:'darkorange', 14:'black'
}

for idx, sample in enumerate(samples_ids):
    adata_sel = adata[(adata.obs['sample'] == sample)]
    for cluster_id in adata_sel.obs[clustering_method].unique():
        cluster_data = adata_sel.obs[adata_sel.obs[clustering_method] == cluster_id]
        colors = clusters_plot[cluster_id] if cluster_id in clusters_plot else "none"
        colors= cluster_data['kmeans_colors'].unique()[0]
        axs[idx].scatter(cluster_data['x_centroid'], cluster_data['y_centroid'], color=colors,  s=0.01, label=cluster_id)
        axs[idx].set_title(f"Sample {sample}")